# 第四章：机械手臂逆运动学 — 交互式可视化

> 台大林沛群教授《机器人学》第四章

本 Notebook 重点说明：
1. 2R 平面机械手臂逆运动学的两组解析解
2. 可达、不可达与工作空间边界
3. 奇异位形的退化现象
4. 简化版 pick-and-place 任务中的 IK 选择

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from ipywidgets import FloatSlider, Dropdown, interactive_output, VBox

%matplotlib inline

plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['font.sans-serif'] = ['Microsoft YaHei', 'SimHei', 'Noto Sans CJK SC', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False


def fk_planar(theta1, theta2, l1=1.2, l2=0.9):
    p0 = np.array([0.0, 0.0])
    p1 = np.array([l1 * np.cos(np.radians(theta1)), l1 * np.sin(np.radians(theta1))])
    t12 = np.radians(theta1 + theta2)
    p2 = p1 + np.array([l2 * np.cos(t12), l2 * np.sin(t12)])
    return np.vstack([p0, p1, p2])


def ik_2r(x, y, l1=1.2, l2=0.9):
    r2 = x**2 + y**2
    c2 = (r2 - l1**2 - l2**2) / (2 * l1 * l2)
    if c2 < -1 or c2 > 1:
        return []
    c2 = np.clip(c2, -1, 1)
    s2_pos = np.sqrt(max(0.0, 1 - c2**2))
    sols = []
    for s2 in (s2_pos, -s2_pos):
        theta2 = np.degrees(np.arctan2(s2, c2))
        k1 = l1 + l2 * c2
        k2 = l2 * s2
        theta1 = np.degrees(np.arctan2(y, x) - np.arctan2(k2, k1))
        sols.append((theta1, theta2))
    return sols


def jacobian_det(theta2, l1=1.2, l2=0.9):
    return l1 * l2 * np.sin(np.radians(theta2))


def draw_arm(ax, pts, title, color='tab:blue'):
    ax.plot(pts[:, 0], pts[:, 1], '-o', color=color, linewidth=3, markersize=8)
    ax.scatter(pts[-1, 0], pts[-1, 1], color='crimson', s=80, zorder=5)
    ax.set_aspect('equal')
    ax.grid(True, alpha=0.3)
    ax.set_title(title)
    ax.set_xlabel('X')
    ax.set_ylabel('Y')


print('Chapter04 工具函数加载完成 ✓')


Chapter04 工具函数加载完成 ✓


---
## Part 1：2R 机械手臂的逆运动学两组解

In [2]:
_x = FloatSlider(value=1.2, min=-2.0, max=2.0, step=0.05, description='target x')
_y = FloatSlider(value=0.8, min=-2.0, max=2.0, step=0.05, description='target y')


def show_ik_solutions(target_x, target_y):
    plt.close('all')
    sols = ik_2r(target_x, target_y)
    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    radius = 2.3

    if not sols:
        ax = axes[0]
        ax.scatter(target_x, target_y, color='red', s=100)
        ax.set_title('目标点不可达')
        ax.set_xlim(-radius, radius)
        ax.set_ylim(-radius, radius)
        ax.set_aspect('equal')
        ax.grid(True, alpha=0.3)
        axes[1].axis('off')
        axes[1].text(0.05, 0.95, '该点不在工作空间内。', transform=axes[1].transAxes, va='top', fontsize=12)
        plt.show()
        return

    labels = ['elbow-up', 'elbow-down']
    colors = ['tab:blue', 'tab:green']
    for ax, (theta1, theta2), label, color in zip(axes, sols, labels, colors):
        pts = fk_planar(theta1, theta2)
        draw_arm(ax, pts, f'{label}\n(theta1={theta1:.1f}°, theta2={theta2:.1f}°)', color=color)
        ax.scatter(target_x, target_y, color='crimson', s=90, marker='x')
        ax.set_xlim(-radius, radius)
        ax.set_ylim(-radius, radius)

    plt.tight_layout()
    plt.show()
    print('IK solutions:')
    for label, (theta1, theta2) in zip(labels, sols):
        print(f'- {label}: theta1={theta1:.3f} deg, theta2={theta2:.3f} deg')


_out1 = interactive_output(show_ik_solutions, {'target_x': _x, 'target_y': _y})
VBox([_x, _y, _out1])


---
## Part 2：可达性与工作空间边界

In [3]:
_px = FloatSlider(value=1.6, min=-2.2, max=2.2, step=0.05, description='probe x')
_py = FloatSlider(value=0.1, min=-2.2, max=2.2, step=0.05, description='probe y')


def show_workspace_and_target(probe_x, probe_y):
    plt.close('all')
    l1, l2 = 1.2, 0.9
    q1s = np.linspace(-180, 180, 120)
    q2s = np.linspace(-180, 180, 120)
    pts = []
    for a in q1s[::3]:
        for b in q2s[::3]:
            pts.append(fk_planar(a, b, l1, l2)[-1])
    pts = np.array(pts)
    reachable = len(ik_2r(probe_x, probe_y, l1, l2)) > 0

    fig, ax = plt.subplots(figsize=(6.8, 6.8))
    ax.scatter(pts[:, 0], pts[:, 1], s=5, alpha=0.14, color='tab:blue')
    ax.scatter(probe_x, probe_y, s=120, color='green' if reachable else 'red', marker='x')
    ax.set_title('工作空间与目标点可达性')
    ax.set_xlabel('X')
    ax.set_ylabel('Y')
    ax.set_aspect('equal')
    ax.grid(True, alpha=0.3)
    ax.set_xlim(-2.3, 2.3)
    ax.set_ylim(-2.3, 2.3)
    plt.show()
    print('Reachable ✓' if reachable else 'Unreachable ×')


_out2 = interactive_output(show_workspace_and_target, {'probe_x': _px, 'probe_y': _py})
VBox([_px, _py, _out2])


---
## Part 3：奇异位形与雅可比退化

In [4]:
_sg_t1 = FloatSlider(value=20, min=-180, max=180, step=5, description='theta1')
_sg_t2 = FloatSlider(value=0, min=-180, max=180, step=5, description='theta2')


def show_singularity(theta1, theta2):
    plt.close('all')
    pts = fk_planar(theta1, theta2)
    detj = jacobian_det(theta2)
    is_singular = abs(detj) < 1e-3

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
    draw_arm(ax1, pts, '当前机构姿态', color='tab:orange')
    ax1.set_xlim(-2.3, 2.3)
    ax1.set_ylim(-2.3, 2.3)

    theta2_grid = np.linspace(-180, 180, 400)
    det_grid = jacobian_det(theta2_grid)
    ax2.plot(theta2_grid, det_grid, color='tab:purple', linewidth=2)
    ax2.axhline(0, color='gray', linestyle='--')
    ax2.axvline(theta2, color='red', linestyle='--')
    ax2.scatter([theta2], [detj], color='red', s=80)
    ax2.set_title('det(J) = l1 l2 sin(theta2)')
    ax2.set_xlabel('theta2 (deg)')
    ax2.set_ylabel('det(J)')
    ax2.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()
    print(f'det(J) = {detj:.6f}')
    print('Singular configuration!' if is_singular else 'Non-singular configuration.')


_out3 = interactive_output(show_singularity, {'theta1': _sg_t1, 'theta2': _sg_t2})
VBox([_sg_t1, _sg_t2, _out3])


---
## Part 4：简化版 Pick-and-Place 任务

In [5]:
_pick_x = FloatSlider(value=1.0, min=-1.8, max=1.8, step=0.05, description='pick x')
_pick_y = FloatSlider(value=0.4, min=-1.8, max=1.8, step=0.05, description='pick y')
_place_x = FloatSlider(value=0.2, min=-1.8, max=1.8, step=0.05, description='place x')
_place_y = FloatSlider(value=1.4, min=-1.8, max=1.8, step=0.05, description='place y')
_branch = Dropdown(options=['elbow-up', 'elbow-down'], value='elbow-up', description='branch')


def show_pick_and_place(pick_x, pick_y, place_x, place_y, branch):
    plt.close('all')
    pick_sols = ik_2r(pick_x, pick_y)
    place_sols = ik_2r(place_x, place_y)
    if not pick_sols or not place_sols:
        fig, ax = plt.subplots(figsize=(6.5, 6.5))
        ax.scatter([pick_x, place_x], [pick_y, place_y], color=['tab:blue', 'tab:orange'], s=120)
        ax.set_title('pick 或 place 点不可达')
        ax.set_aspect('equal')
        ax.grid(True, alpha=0.3)
        ax.set_xlim(-2.3, 2.3)
        ax.set_ylim(-2.3, 2.3)
        plt.show()
        return

    idx = 0 if branch == 'elbow-up' else 1
    pick_cfg = pick_sols[idx]
    place_cfg = place_sols[idx]
    pick_pts = fk_planar(*pick_cfg)
    place_pts = fk_planar(*place_cfg)
    transit = np.array([
        [pick_x, pick_y],
        [pick_x, pick_y + 0.3],
        [place_x, place_y + 0.3],
        [place_x, place_y],
    ])

    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    draw_arm(axes[0], pick_pts, f'Pick pose ({branch})', color='tab:blue')
    axes[0].scatter(pick_x, pick_y, color='crimson', s=90, marker='x')
    axes[0].set_xlim(-2.3, 2.3)
    axes[0].set_ylim(-2.3, 2.3)

    draw_arm(axes[1], place_pts, f'Place pose ({branch})', color='tab:green')
    axes[1].plot(transit[:, 0], transit[:, 1], '--', color='gray', linewidth=2, label='task path')
    axes[1].scatter(transit[:, 0], transit[:, 1], color='black', s=30)
    axes[1].scatter(place_x, place_y, color='crimson', s=90, marker='x')
    axes[1].legend()
    axes[1].set_xlim(-2.3, 2.3)
    axes[1].set_ylim(-2.3, 2.3)

    plt.tight_layout()
    plt.show()
    print(f'Pick IK: theta1={pick_cfg[0]:.3f}, theta2={pick_cfg[1]:.3f}')
    print(f'Place IK: theta1={place_cfg[0]:.3f}, theta2={place_cfg[1]:.3f}')


_out4 = interactive_output(show_pick_and_place, {
    'pick_x': _pick_x,
    'pick_y': _pick_y,
    'place_x': _place_x,
    'place_y': _place_y,
    'branch': _branch,
})
VBox([_pick_x, _pick_y, _place_x, _place_y, _branch, _out4])


---
## Part 5：批量导出静态图

运行下面代码单元后，执行 `export_all_chapter04_figures()` 可导出本章常用图片到 `Robotics_NTU/images/`。

In [6]:
def _resolve_output_dir():
    cwd = Path.cwd()
    for base in (cwd, cwd / 'Robotics_NTU'):
        if (base / 'Chapter04_visualization.ipynb').exists():
            out = base / 'images'
            out.mkdir(parents=True, exist_ok=True)
            return out
    out = cwd / 'images'
    out.mkdir(parents=True, exist_ok=True)
    return out


OUTPUT_DIR = _resolve_output_dir()
print(f'Export directory: {OUTPUT_DIR}')


def export_ch04_two_solutions():
    target_x, target_y = 1.2, 0.8
    sols = ik_2r(target_x, target_y)
    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    labels = ['elbow-up', 'elbow-down']
    colors = ['tab:blue', 'tab:green']
    for ax, (theta1, theta2), label, color in zip(axes, sols, labels, colors):
        pts = fk_planar(theta1, theta2)
        draw_arm(ax, pts, label, color=color)
        ax.scatter(target_x, target_y, color='crimson', s=90, marker='x')
        ax.set_xlim(-2.3, 2.3)
        ax.set_ylim(-2.3, 2.3)
    path = OUTPUT_DIR / 'ch04_two_ik_solutions.png'
    fig.savefig(path, dpi=150, bbox_inches='tight')
    plt.close(fig)
    return path


def export_ch04_workspace():
    q1s = np.linspace(-180, 180, 120)
    q2s = np.linspace(-180, 180, 120)
    pts = []
    for a in q1s[::3]:
        for b in q2s[::3]:
            pts.append(fk_planar(a, b)[-1])
    pts = np.array(pts)
    fig, ax = plt.subplots(figsize=(6.5, 6.5))
    ax.scatter(pts[:, 0], pts[:, 1], s=5, alpha=0.14, color='tab:blue')
    ax.scatter(1.6, 0.1, s=120, color='green', marker='x')
    ax.scatter(2.2, 0.4, s=120, color='red', marker='x')
    ax.set_title('Reachable vs Unreachable Targets')
    ax.set_xlabel('X')
    ax.set_ylabel('Y')
    ax.set_aspect('equal')
    ax.grid(True, alpha=0.3)
    ax.set_xlim(-2.3, 2.3)
    ax.set_ylim(-2.3, 2.3)
    path = OUTPUT_DIR / 'ch04_workspace_reachability.png'
    fig.savefig(path, dpi=150, bbox_inches='tight')
    plt.close(fig)
    return path


def export_ch04_pick_and_place():
    pick_cfg = ik_2r(1.0, 0.4)[0]
    place_cfg = ik_2r(0.2, 1.4)[0]
    pick_pts = fk_planar(*pick_cfg)
    place_pts = fk_planar(*place_cfg)
    transit = np.array([[1.0, 0.4], [1.0, 0.7], [0.2, 1.7], [0.2, 1.4]])
    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    draw_arm(axes[0], pick_pts, 'Pick pose', color='tab:blue')
    axes[0].scatter(1.0, 0.4, color='crimson', s=90, marker='x')
    axes[0].set_xlim(-2.3, 2.3)
    axes[0].set_ylim(-2.3, 2.3)
    draw_arm(axes[1], place_pts, 'Place pose', color='tab:green')
    axes[1].plot(transit[:, 0], transit[:, 1], '--', color='gray', linewidth=2, label='task path')
    axes[1].scatter(transit[:, 0], transit[:, 1], color='black', s=30)
    axes[1].scatter(0.2, 1.4, color='crimson', s=90, marker='x')
    axes[1].legend()
    axes[1].set_xlim(-2.3, 2.3)
    axes[1].set_ylim(-2.3, 2.3)
    path = OUTPUT_DIR / 'ch04_pick_and_place.png'
    fig.savefig(path, dpi=150, bbox_inches='tight')
    plt.close(fig)
    return path


def export_all_chapter04_figures():
    paths = [
        export_ch04_two_solutions(),
        export_ch04_workspace(),
        export_ch04_pick_and_place(),
    ]
    print('Exported files:')
    for path in paths:
        print(f'- {path}')
    return paths


# export_all_chapter04_figures()


Export directory: d:\EI-Beginner\Robotics_NTU\images
